# Semantic ID Generation — Part 1: Foundations & Baselines

**Based on**: *Transformer Memory as a Differentiable Search Index* (Tay et al., NeurIPS 2022),
*Product Quantization for Nearest Neighbor Search* (Jégou et al., TPAMI 2011),
*Recommender Systems with Generative Retrieval* (Rajput et al., NeurIPS 2023 — TIGER)

---

In generative retrieval and recommendation, a model autoregressively decodes an item's identifier
given a query. The choice of ID scheme determines what the model can learn: random IDs force
memorisation of every item independently, while **semantic IDs** assign structured, meaningful
identifiers so that semantically similar items share prefixes or sub-codes.

This three-part tutorial explains four approaches, from simplest to state-of-the-art:

1. **Random / Atomic IDs** — arbitrary integers, no semantic structure (baseline) — *this part*
2. **Hierarchical K-Means** — recursive clustering; ID = tree path (DSI) — Part 2
3. **Product Quantization (PQ)** — independent subspace codebooks (FAISS) — Part 2
4. **RQ-VAE** — learned residual quantization via autoencoder (TIGER) — Part 3

A shared toy dataset and a *semantic coherence* metric let us compare all four on equal footing.

**Exercises**: cells marked `# YOUR CODE HERE` require you to fill in the missing implementation.
Each exercise is followed by an assertion cell that verifies correctness.
Solutions are at the end of each part.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from torch import optim

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

COLORS = ["steelblue", "darkorange", "seagreen", "crimson", "mediumpurple"]

| Symbol | Meaning |
|--------|---------|
| $\mathbf{e}_i \in \mathbb{R}^D$ | Embedding of item $i$ |
| $\text{id}(i) = (c_1, c_2, \ldots, c_L)$ | Semantic ID: tuple of $L$ tokens |
| $N$ | Number of items |
| $K$ | Codebook size (branching factor) |
| $M$ | Number of subspaces (PQ) or quantization levels (RQ-VAE) |
| $\text{SC}$ | Semantic coherence (defined in §1) |

---
## 1. Toy Dataset

We generate $N = 200$ items from 5 Gaussian clusters in two variants:

- **2D** ($D=2$): for scatter-plot visualisation throughout the notebook
- **64D** ($D=64$): for the quantisation approaches where higher dimensionality is needed

Both share the same cluster assignments so results are directly comparable.

### Semantic coherence metric

To quantify how well an ID scheme preserves neighbourhood structure, we define:

$$\text{SC}(f) = \frac{1}{N}\sum_{i=1}^{N} \frac{\bigl|\{j : j \in \text{kNN}_k(i) \;\land\; \text{prefix}_1(f(i)) = \text{prefix}_1(f(j))\}\bigr|}{k}$$

where $f$ maps items to IDs and $\text{kNN}_k(i)$ are the $k$ nearest neighbours by
embedding distance. $\text{prefix}_1$ denotes the first token of the ID.
SC $\in [0, 1]$: higher means neighbours share ID prefixes more often.

In [ ]:
N_ITEMS = 200
N_CLUSTERS = 5
ITEMS_PER_CLUSTER = N_ITEMS // N_CLUSTERS

# Cluster centres
centers_2d = np.array([
    [-3.0, -3.0],
    [ 3.0, -3.0],
    [ 0.0,  3.0],
    [-3.0,  3.0],
    [ 3.0,  3.0],
])

# Generate 2D embeddings
embeddings_2d = np.vstack([
    np.random.randn(ITEMS_PER_CLUSTER, 2) * 0.8 + centers_2d[idx]
    for idx in range(N_CLUSTERS)
])
labels = np.repeat(np.arange(N_CLUSTERS), ITEMS_PER_CLUSTER)

# Generate 64D embeddings with the same cluster structure
centers_64d = np.random.randn(N_CLUSTERS, 64) * 3.0
embeddings_64d = np.vstack([
    np.random.randn(ITEMS_PER_CLUSTER, 64) * 0.5 + centers_64d[idx]
    for idx in range(N_CLUSTERS)
])

print(f"2D embeddings : {embeddings_2d.shape}")
print(f"64D embeddings: {embeddings_64d.shape}")
print(f"Clusters      : {N_CLUSTERS}, {ITEMS_PER_CLUSTER} items each")

In [ ]:
def semantic_coherence(
    embeddings: np.ndarray,
    ids: np.ndarray,
    k: int = 10,
) -> float:
    """Fraction of k-NN that share the first token of their ID.

    Args:
        embeddings: (N, D) array of item embeddings.
        ids: (N,) or (N, L) array of IDs. First column used as prefix.
        k: number of nearest neighbours.

    Returns:
        SC in [0, 1].
    """
    ids = np.asarray(ids)
    if ids.ndim == 2:
        prefixes = ids[:, 0]
    else:
        prefixes = ids

    dists = cdist(embeddings, embeddings, metric="euclidean")
    np.fill_diagonal(dists, np.inf)
    n_items = len(embeddings)
    sc_sum = 0.0
    for idx in range(n_items):
        neighbours = np.argsort(dists[idx])[:k]
        matches = np.sum(prefixes[neighbours] == prefixes[idx])
        sc_sum += matches / k
    return sc_sum / n_items

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for cluster_idx in range(N_CLUSTERS):
    mask = labels == cluster_idx
    ax.scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=COLORS[cluster_idx], label=f"Cluster {cluster_idx}", alpha=0.7, s=30,
    )
ax.set_title("Toy Dataset: 200 Items from 5 Gaussian Clusters")
ax.set_xlabel("$e_1$")
ax.set_ylabel("$e_2$")
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Approach 1: Random / Atomic IDs (Baseline)

The simplest scheme: each item receives an arbitrary integer
$\text{id}(i) \in \{0, 1, \ldots, N-1\}$ with no semantic structure.
This is the default in standard recommender systems (collaborative filtering,
matrix factorisation).

In generative retrieval, atomic IDs mean the model must memorise each item
independently — the vocabulary equals the corpus size, and no generalisation
across similar items is possible.

A hash-based variant:

$$\text{id}(i) = h(\mathbf{e}_i) \bmod N$$

The mapping is effectively a random permutation — no relationship between
$\text{id}(i)$ and $\text{id}(j)$ even when $\|\mathbf{e}_i - \mathbf{e}_j\|$ is small.

In [ ]:
# Sequential IDs, randomly shuffled
ids_random = np.arange(N_ITEMS)
np.random.shuffle(ids_random)

# Hash-based IDs
ids_hash = np.array([hash(embeddings_2d[idx].tobytes()) % N_ITEMS for idx in range(N_ITEMS)])

sc_random = semantic_coherence(embeddings_2d, ids_random)
sc_hash = semantic_coherence(embeddings_2d, ids_hash)

print(f"SC (shuffled sequential): {sc_random:.3f}")
print(f"SC (hash-based)         : {sc_hash:.3f}")
print(f"Expected by chance      : ~{1/N_CLUSTERS:.3f} (1/{N_CLUSTERS} clusters)")

### Exercise 1

Implement a hash-based ID assignment. Given an array of embeddings and a number of
buckets, assign each item an integer ID by hashing its embedding vector.

**Hint**: `hash(embeddings[idx].tobytes()) % num_buckets`

In [ ]:
def assign_hash_ids(embeddings: np.ndarray, num_buckets: int) -> np.ndarray:
    """Assign each item a hash-based ID.

    Args:
        embeddings: (N, D) array of item embeddings.
        num_buckets: number of distinct ID values.

    Returns:
        ids: (N,) integer array with values in [0, num_buckets).
    """
    ids = np.zeros(len(embeddings), dtype=np.int64)
    for idx in range(len(embeddings)):
        # YOUR CODE HERE: hash the embedding and mod by num_buckets
        pass
    return ids

In [ ]:
test_ids = assign_hash_ids(embeddings_2d, num_buckets=200)
assert test_ids.shape == (N_ITEMS,), f"Expected shape ({N_ITEMS},), got {test_ids.shape}"
assert test_ids.min() >= 0 and test_ids.max() < 200, "IDs must be in [0, num_buckets)"
assert len(np.unique(test_ids)) > 1, "IDs should not all be the same"
sc_ex1 = semantic_coherence(embeddings_2d, test_ids)
print(f"Semantic coherence of your hash IDs: {sc_ex1:.3f}")
print("Exercise 1 passed!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: ground-truth clusters
for cluster_idx in range(N_CLUSTERS):
    mask = labels == cluster_idx
    axes[0].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=COLORS[cluster_idx], alpha=0.7, s=30,
    )
axes[0].set_title("Ground-Truth Clusters")
axes[0].set_xlabel("$e_1$")
axes[0].set_ylabel("$e_2$")

# Right: colored by random hash IDs (bucketed into 5 groups for visibility)
hash_groups = ids_hash % N_CLUSTERS
for group_idx in range(N_CLUSTERS):
    mask = hash_groups == group_idx
    axes[1].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=COLORS[group_idx], alpha=0.7, s=30,
    )
axes[1].set_title("Random Hash IDs (mod 5)")
axes[1].set_xlabel("$e_1$")
axes[1].set_ylabel("$e_2$")

plt.suptitle("Random IDs impose no spatial structure", y=1.02)
plt.tight_layout()
plt.show()

**Pros**: Simple, $O(1)$ assignment, unique IDs guaranteed.

**Cons**: No generalisation — the model must memorise each item independently;
vocabulary scales linearly with corpus size $N$;
no shared structure for prefix-based beam search in generative retrieval.

What if we *assigned* IDs based on where items sit in embedding space?

---
## Appendix: Exercise Solutions

Expand the cells below only after attempting the exercises.

In [ ]:
# Solution 1: assign_hash_ids
def assign_hash_ids_solution(embeddings: np.ndarray, num_buckets: int) -> np.ndarray:
    ids = np.zeros(len(embeddings), dtype=np.int64)
    for idx in range(len(embeddings)):
        ids[idx] = hash(embeddings[idx].tobytes()) % num_buckets
    return ids

print("Solution 1: ids[idx] = hash(embeddings[idx].tobytes()) % num_buckets")

---
**Next**: [Part 2 — Clustering & Quantization](./part2-clustering-and-quantization.ipynb) covers
Hierarchical K-Means and Product Quantization.